# Student Performance Prediction: Model Training

This notebook details the model selection, pipeline engineering, training, hyperparameter tuning, and evaluation stages. Our objective is to build a regression model to predict `math_score` based on demographic and academic attributes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling and metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

## 1. Import Data

We read in the dataset and standardize column naming structures.

In [ ]:
df = pd.read_csv('../data/stud.csv')
df.columns = [col.strip().replace(' ', '_').replace('/', '_') for col in df.columns]
df.head()

## 2. Define X (Features) and y (Target Variable)

Since we are predicting a continuous value (`math_score`), this is a **supervised regression** task. X will hold the independent student demographics and features, while y will be the target variable.

In [ ]:
X = df.drop(columns=['math_score'], axis=1)
y = df['math_score']

print("X shape:", X.shape)
print("y shape:", y.shape)

## 3. Feature Transformation Pipeline

Before feeding data into ML models, we must preprocess the inputs:
1. **Categorical Features**: Machine learning algorithms operate on numerical vectors. We map categorical text (gender, race, etc.) to one-hot binary flags.
2. **Numerical Features**: Numerical values (`reading_score`, `writing_score`) are scaled using `StandardScaler` to have a mean of 0 and standard deviation of 1. Scaling ensures distance-based or regularized models do not weight features with larger absolute scales more heavily.

We combine these transformers using a `ColumnTransformer` object.

In [ ]:
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features),
    ]
)

In [ ]:
# Fit and transform X
X = preprocessor.fit_transform(X)
X.shape

## 4. Train-Test Split

We split the dataset into an **80% training set** (to teach the models) and a **20% testing set** (to validate on unseen data) with `random_state=42` to ensure reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 5. Model Evaluation Definition

We evaluate predictions using three standard metrics:
- **Mean Absolute Error (MAE)**: Average of absolute prediction errors. Easy to interpret.
- **Root Mean Squared Error (RMSE)**: Square root of average squared differences. Penalizes larger errors.
- **R² Score (Coefficient of Determination)**: Proportion of target variance explained by features. Closer to 1.0 is better.

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

## 6. Training & Comparing Multiple Regression Algorithms

We test several modeling families (linear models, regularized regressors, distance-based neighbors, tree methods, and boosting ensembles) to identify the best architecture.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoost Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(name)
    model_list.append(name)
    
    print('Model performance for Training set:')
    print(f"- RMSE: {model_train_rmse:.4f}")
    print(f"- MAE: {model_train_mae:.4f}")
    print(f"- R2 Score: {model_train_r2:.4f}")
    print('----------------------------------')
    print('Model performance for Test set:')
    print(f"- RMSE: {model_test_rmse:.4f}")
    print(f"- MAE: {model_test_mae:.4f}")
    print(f"- R2 Score: {model_test_r2:.4f}")
    r2_list.append(model_test_r2)
    print('='*35)
    print('\n')

## 7. Model Performance Summary

We compile our test evaluation results into a sorted table to identify the best model.

In [ ]:
results_df = pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score'])
results_df = results_df.sort_values(by=["R2_Score"], ascending=False)
results_df

## 8. Analyzing Predictions from the Best Model

Let's select the best model (e.g., Linear Regression/Lasso/Ridge), fit it again, and inspect the differences between the actual and predicted math scores.

In [ ]:
best_model_name = results_df.iloc[0]['Model Name']
print(f"Selected Best Model: {best_model_name}\n")

best_model = models[best_model_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

# Build prediction details dataframe
pred_df = pd.DataFrame({'Actual_Value': y_test, 'Predicted_Value': y_pred, 'Difference': y_test - y_pred})
pred_df.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(x=y_test, y=y_pred, ci=None, color="blue", scatter_kws={"alpha":0.5})
plt.xlabel('Actual Math Score')
plt.ylabel('Predicted Math Score')
plt.title('Actual vs Predicted Math Scores')
plt.show()